# Sentiment-position results: 2026-09-18_20-22_CDT

This read-only Colab notebook examines the completed sentiment-position run `2026-09-18_20-22_CDT`. It produces the three causal-metric position plots for both models and lists every layer selected using ToyMovieReview ADVERB performance.

> Selection protocol: ADVERB `logit_flip_percent` chooses one layer independently for every model × fitting method × fitting position. The selected direction is then evaluated at that frozen layer on ADVERB, ADJ, and SST.

## Read-only contract

The notebook mounts Google Drive and reads the saved CSV files. It does not alter the run, reselect layers, load a language model, require a GPU, or request a Hugging Face token. Plots and tables are displayed in notebook memory only.

## 1. Fixed run and repository settings

In [ ]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optionally pin the reporting-code commit.
DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
RUN_ID = "2026-09-18_20-22_CDT"

## 2. Install and verify the reporting package

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
from sentiment_geometry.reporting import (
    MODEL_ORDER,
    SentimentPositionReportData,
)

expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )
print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)

## 3. Mount Drive and open the fixed run

In [ ]:
import json

import pandas as pd
from google.colab import drive
from IPython.display import Markdown, display

DRIVE_MOUNT_ROOT = Path("/content/drive")
if not (DRIVE_MOUNT_ROOT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)
else:
    print(f"Google Drive is already mounted at {DRIVE_MOUNT_ROOT}")

RUN_ROOT = (
    Path(DRIVE_STORAGE_ROOT)
    / "sentiment-position-comparison"
    / "runs"
    / RUN_ID
)
RESULTS_ROOT = RUN_ROOT / "results"
MANIFEST_PATH = RUN_ROOT / "run_manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(MANIFEST_PATH)
if not RESULTS_ROOT.is_dir():
    raise FileNotFoundError(RESULTS_ROOT)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if manifest.get("status") != "completed":
    raise RuntimeError(
        f"Run {RUN_ID} is not marked completed: {manifest.get('status')!r}"
    )
display(pd.Series(manifest, name="value").to_frame())
print("Selected read-only run:", RUN_ROOT)

## 4. Load and validate frozen-layer results

In [ ]:
from itertools import product

POSITION_ORDER = ("adjective", "verb", "summary", "final")
POSITION_LABELS = ("ADJ", "VRB", "SUM", "END")
METHOD_ORDER = ("mean_diff", "logistic_regression", "das")
DATASET_ORDER_EXPECTED = ("toy_adverbs", "toy_adjectives", "sst")

report = SentimentPositionReportData.load(RESULTS_ROOT)
if report.has_training_causal_metrics:
    raise RuntimeError("Unexpected toy_train causal metrics were found.")
if tuple(report.models) != tuple(MODEL_ORDER):
    raise RuntimeError(f"Expected models {MODEL_ORDER}; got {report.models}")
if report.evaluation_datasets != DATASET_ORDER_EXPECTED:
    raise RuntimeError(
        f"Expected datasets {DATASET_ORDER_EXPECTED}; got {report.evaluation_datasets}"
    )

cell_columns = ["model", "method", "fit_position", "dataset"]
if report.selected_metrics.duplicated(cell_columns).any():
    duplicates = report.selected_metrics[
        report.selected_metrics.duplicated(cell_columns, keep=False)
    ][cell_columns]
    raise RuntimeError(f"Duplicate selected-layer cells found:\n{duplicates}")
expected_cells = set(
    product(MODEL_ORDER, METHOD_ORDER, POSITION_ORDER, DATASET_ORDER_EXPECTED)
)
actual_cells = set(
    report.selected_metrics[cell_columns].itertuples(index=False, name=None)
)
missing_cells = sorted(expected_cells - actual_cells)
unexpected_cells = sorted(actual_cells - expected_cells)
if missing_cells or unexpected_cells:
    raise RuntimeError(
        f"Incomplete selected-layer grid. Missing={missing_cells}; "
        f"unexpected={unexpected_cells}"
    )
display(report.dataset_summary.reset_index(drop=True))
print(f"Validated {len(report.selected_metrics):,} frozen-layer metric rows.")
print(f"Validated {len(report.layer_selection):,} ADVERB layer selections.")

## 5. Causal metrics across ADJ → VRB → SUM → END

For each model and metric, rows are fitting methods and columns are evaluation datasets. Positions are converted to the explicit numeric indices `0, 1, 2, 3` before plotting. Every panel therefore connects exactly `ADJ → VRB → SUM → END`, regardless of the row order in the CSV file.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_LABELS = {
    "gpt2-small": "GPT-2 Small",
    "qwen-0.6b": "Qwen3-0.6B Base",
}
METHOD_LABELS = {
    "mean_diff": "Mean Difference",
    "logistic_regression": "Logistic Regression",
    "das": "DAS",
}
DATASET_LABELS = {
    "toy_adverbs": "ToyMovieReview(ADVRB)",
    "toy_adjectives": "ToyMovieReview(ADJ)",
    "sst": "SST",
}
METRIC_SPECS = {
    "logit_difference_percent": "Logit difference (%)",
    "logit_flip_percent": "Logit flip (%)",
    "sign_flip_percent": "Literal sign flip (%)",
}
METHOD_COLORS = {
    "mean_diff": "#0072B2",
    "logistic_regression": "#D55E00",
    "das": "#009E73",
}
POSITION_TO_INDEX = dict(zip(POSITION_ORDER, range(len(POSITION_ORDER))))

plot_data = report.selected_metrics.copy()
plot_data["position_index"] = plot_data["fit_position"].map(POSITION_TO_INDEX)
if plot_data["position_index"].isna().any():
    unknown = sorted(plot_data.loc[plot_data["position_index"].isna(), "fit_position"].unique())
    raise RuntimeError(f"Unknown fitting positions: {unknown}")
plot_data["position_index"] = plot_data["position_index"].astype(int)
plot_data = plot_data.sort_values(
    ["model", "method", "dataset", "position_index"], kind="stable"
)

sns.set_theme(style="whitegrid", context="notebook")
for model in MODEL_ORDER:
    for metric_column, metric_label in METRIC_SPECS.items():
        figure, axes = plt.subplots(
            len(METHOD_ORDER),
            len(DATASET_ORDER_EXPECTED),
            figsize=(14, 10),
            sharex=True,
            sharey=True,
            squeeze=False,
        )
        for row_index, method in enumerate(METHOD_ORDER):
            for column_index, dataset in enumerate(DATASET_ORDER_EXPECTED):
                axis = axes[row_index, column_index]
                panel = plot_data[
                    (plot_data["model"] == model)
                    & (plot_data["method"] == method)
                    & (plot_data["dataset"] == dataset)
                ].sort_values("position_index", kind="stable")
                observed_order = tuple(panel["fit_position"])
                if observed_order != POSITION_ORDER:
                    raise RuntimeError(
                        f"Incorrect position order for {model}/{method}/{dataset}: "
                        f"{observed_order}"
                    )
                axis.plot(
                    panel["position_index"],
                    panel[metric_column],
                    marker="o",
                    linewidth=2.2,
                    markersize=6,
                    color=METHOD_COLORS[method],
                )
                axis.set_xticks(range(len(POSITION_ORDER)), POSITION_LABELS)
                axis.axhline(0, color="black", linewidth=0.7, alpha=0.5)
                if row_index == 0:
                    axis.set_title(DATASET_LABELS[dataset])
                if column_index == 0:
                    axis.set_ylabel(metric_label)
                if row_index == len(METHOD_ORDER) - 1:
                    axis.set_xlabel("Activation position")
                if column_index == len(DATASET_ORDER_EXPECTED) - 1:
                    axis.annotate(
                        METHOD_LABELS[method],
                        xy=(1.04, 0.5),
                        xycoords="axes fraction",
                        rotation=-90,
                        va="center",
                        ha="left",
                        fontsize=11,
                    )
        figure.suptitle(
            f"{MODEL_LABELS[model]}: {metric_label} across fitting positions",
            fontsize=16,
        )
        figure.tight_layout(rect=(0, 0, 0.97, 0.96))
        plt.show()
        plt.close(figure)

## 6. Layers selected by ADVERB performance

Each cell below shows `Lxx (score%)`, where `Lxx` is the selected residual boundary and `score%` is its ADVERB `logit_flip_percent`. The notebook also verifies that every SST row was evaluated at the corresponding selected layer.

In [ ]:
selection = report.layer_selection.copy()
if set(selection["selection_dataset"]) != {"toy_adverbs"}:
    raise RuntimeError("Layers were not selected exclusively on toy_adverbs.")
if set(selection["selection_metric"]) != {"logit_flip_percent"}:
    raise RuntimeError("Unexpected layer-selection metric.")
selection_keys = ["model", "method", "fit_position"]
if selection.duplicated(selection_keys).any():
    raise RuntimeError("Duplicate layer-selection rows were found.")

sst_layers = report.selected_metrics[report.selected_metrics["dataset"] == "sst"][
    [*selection_keys, "layer"]
].merge(
    selection[[*selection_keys, "selected_layer"]],
    on=selection_keys,
    how="left",
    validate="one_to_one",
)
mismatched_sst = sst_layers[sst_layers["layer"] != sst_layers["selected_layer"]]
if not mismatched_sst.empty:
    raise RuntimeError(f"SST layer mismatch found:\n{mismatched_sst}")

selection["fit_position"] = pd.Categorical(
    selection["fit_position"], categories=POSITION_ORDER, ordered=True
)
selection["method"] = pd.Categorical(
    selection["method"], categories=METHOD_ORDER, ordered=True
)
selection["layer_and_score"] = selection.apply(
    lambda row: (
        f"L{int(row['selected_layer']):02d} "
        f"({float(row['selection_value_percent']):.1f}%)"
    ),
    axis=1,
)
for model in MODEL_ORDER:
    model_selection = selection[selection["model"] == model].sort_values(
        ["method", "fit_position"], kind="stable"
    )
    table = model_selection.pivot(
        index="method", columns="fit_position", values="layer_and_score"
    ).reindex(index=METHOD_ORDER, columns=POSITION_ORDER)
    table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
    table.columns = POSITION_LABELS
    display(Markdown(f"### {MODEL_LABELS[model]}"))
    display(table.style.set_properties(**{"text-align": "center"}))
print("Confirmed: every SST result uses its corresponding ADVERB-selected layer.")

## Finished

The run remains unchanged on Google Drive. Restart the Colab runtime to discard the in-memory tables and figures.